# gnn_mech: Colab driver

Runs `gnn_mech` experiments (over-squashing vs attention dilution in hybrid GNNs on
LRGB Peptides-func) on a Colab GPU. Data and results go to Google Drive.

**Runs resume on their own.** Every run writes `checkpoint.pt` to Drive every
`train.checkpoint_every` epochs. After a Colab disconnect, rerun the setup cells and then
the same run cell: training continues from the last checkpoint. Runs that already have
`metrics.json` are skipped (pass `--force` to retrain).

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Get the code

Clones the repo, or pulls if it is already there. If the repo is private, create a GitHub
personal access token (read access to the repo is enough), store it as a Colab secret named
`GITHUB_TOKEN` (key icon in the left sidebar) and set `USE_TOKEN = True`. Never paste the
token into the notebook itself.

In [ ]:
import os, subprocess

BRANCH = 'claude/bold-thompson-1dbz9u'
REPO_DIR = '/content/message-passing-in-gnns'
USE_TOKEN = False

url = 'https://github.com/djdhawal/message-passing-in-gnns.git'
if USE_TOKEN:
    from google.colab import userdata
    url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/djdhawal/message-passing-in-gnns.git"

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, url, REPO_DIR], check=True)
%cd {REPO_DIR}
!git log -1 --oneline

## 4. Install

In [ ]:
!pip install -q torch_geometric
!pip install -q -e .

## 5. Paths on Drive

The dataset is downloaded once into `DATA_ROOT`; every run writes to
`OUT_DIR/<exp_name>/<config_hash>/seed<seed>/`.

In [ ]:
DRIVE = '/content/drive/MyDrive/Thesis/gnn_mech'
DATA_ROOT = f'{DRIVE}/data'
OUT_DIR = f'{DRIVE}/runs'
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
COMMON = f'out_dir={OUT_DIR} data.root={DATA_ROOT}'
print(COMMON)

## 6. Smoke test (synthetic data, a few minutes)

In [ ]:
!python -m gnn_mech.run --config configs/smoke.yaml --seed 0 {COMMON}

## 7. Phase 1 step 7: pilot equivalence check

Hybrid with alpha = 0.5, 96 hidden, 4 layers, matching the pilot LRGB notebook's GPS.
Target: test AP >= 0.652 on seed 0.

In [ ]:
!python -m gnn_mech.run --config configs/pilot_equivalence.yaml --seed 0 {COMMON}

### Sanity runs at the ends of the spectrum: alpha = 0 (pure GINE) and alpha = 1 (pure attention)

In [ ]:
!python -m gnn_mech.run --config configs/pilot_equivalence.yaml --seed 0 model.alpha=0.0 {COMMON}
!python -m gnn_mech.run --config configs/pilot_equivalence.yaml --seed 0 model.alpha=1.0 {COMMON}

## 8. Results table: mean ± std over seeds

In [ ]:
import glob, json
import pandas as pd

rows = []
for path in glob.glob(f'{OUT_DIR}/*/*/seed*/metrics.json'):
    run_dir = os.path.dirname(path)
    with open(path) as f:
        m = json.load(f)
    with open(os.path.join(run_dir, 'config.json')) as f:
        cfg = json.load(f)
    row = {'exp': cfg['exp_name'], 'hash': m['config_hash'], 'arch': cfg['model']['arch'],
           'alpha': cfg['model']['alpha'], 'seed': cfg['seed'], 'best_epoch': m['best_epoch'],
           'epochs_run': m['epochs_run'], 'n_params': m['n_params'], 'hours': m['seconds'] / 3600}
    row.update({f'val_{k}': v for k, v in m['val'].items()})
    row.update({f'test_{k}': v for k, v in m['test'].items()})
    rows.append(row)

df = pd.DataFrame(rows)
if df.empty:
    print('no finished runs yet')
else:
    metric_cols = [c for c in df.columns if c.startswith(('val_', 'test_')) and not c.endswith('loss')]
    summary = (df.groupby(['exp', 'hash', 'arch', 'alpha'])
                 .agg(n_seeds=('seed', 'nunique'), n_params=('n_params', 'first'),
                      **{f'{c}_mean': (c, 'mean') for c in metric_cols},
                      **{f'{c}_std': (c, 'std') for c in metric_cols})
                 .reset_index())
    for c in metric_cols:
        summary[c] = summary.apply(lambda r: f"{r[c + '_mean']:.4f} ± {0 if pd.isna(r[c + '_std']) else r[c + '_std']:.4f}", axis=1)
    display(summary[['exp', 'hash', 'arch', 'alpha', 'n_seeds', 'n_params'] + metric_cols])

## 9. Phase 2: Peptides-func re-baseline (after the G0 check passes)

Twelve runs: alpha = 0 / 0.5 / 1 hybrids and the Tönshoff-aligned GCN, seeds 0-2, each
followed by Jacobian, entropy and range measurements and `preds_val.npz` / `preds_test.npz`.
About 4 h of training plus 1 h of measurement; it spans two or three Colab sessions.
**After a disconnect, rerun the setup cells (1-5) and then the sweep cell:** finished runs
are skipped and an interrupted run resumes from its last checkpoint.

First the run list, each run's status and the estimated remaining hours (nothing is trained):

In [ ]:
!python -m gnn_mech.sweep configs/sweeps/phase2.yaml --dry-run {COMMON}

In [ ]:
!python -m gnn_mech.sweep configs/sweeps/phase2.yaml {COMMON}

### Phase 2 analysis

Tables (CSV), `summary.json` and figures (PDF + PNG) go to `REPORT_DIR` on Drive.
`notebooks/phase2_analysis.ipynb` shows the same results interactively.

In [ ]:
REPORT_DIR = f'{DRIVE}/reports/phase2'
!python -m gnn_mech.analysis {OUT_DIR} --exp peptides_func_alpha peptides_func_gcn --out {REPORT_DIR}

## 10. Phase 2 Reddit track (CPU is fine)

The graph is built from SNAP `soc-redditHyperlinks-body.tsv` (downloaded once into
`REDDIT_ROOT`). If SNAP is unreachable, point `REDDIT_PT` at the pilot's
`reddit_base_graph.pt` on Drive. Step 1 ranks candidate targets (sentiment and LIWC
columns) by how much 1..4-hop neighbourhood features add over the node's own features;
pick the target from its output (`selected`, or another row of the table), then run the
main (distance split) and control (random split) experiments, 30 seeds per model.

In [ ]:
REDDIT_ROOT = f'{DRIVE}/data/reddit'
REDDIT_PT = '/content/drive/MyDrive/Thesis /reddit_base_graph.pt'   # fallback if SNAP is unreachable
REDDIT_ARGS = f'--root {REDDIT_ROOT}' + (f' --cached-pt "{REDDIT_PT}"' if os.path.exists(REDDIT_PT) else '')
!python -m gnn_mech.reddit.target_select {REDDIT_ARGS} --out {OUT_DIR}/reddit/target_selection.json

In [ ]:
import json
sel = json.load(open(f'{OUT_DIR}/reddit/target_selection.json'))
print(sel['reason'])
TARGET = sel['selected']   # or set by hand from sel['table']
print('target:', TARGET)

In [ ]:
!python -m gnn_mech.reddit.run --target {TARGET} --seeds 0:30 {REDDIT_ARGS} --out-dir {OUT_DIR}